# Example 7.4 / 1: Minimize the Largest Deviation

Companion notebook for **7.4 Linearizing abs, max and min** in the AMPL lab Documentation, section "Minimize the Largest Deviation". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Minimize the Largest Deviation

Choose one value `x` as close as possible to targets 2, 5 and 9, measured by the worst deviation.
`worst` must exceed `abs(x - target[i])` for every target, and minimizing it pushes it down to
the largest deviation:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set I;                            # targets to approach
param target {I};                 # target value
var x;                            # single value chosen for all targets
var worst >= 0;                   # largest deviation from any target

subject to PositiveError {i in I}: worst >= x - target[i];  # x above a target
subject to NegativeError {i in I}: worst >= target[i] - x;  # x below a target
minimize MinimaxError: worst;     # pushes worst down to the true maximum

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["I"] = ["a", "b", "c"]
ampl.param["target"] = {"a": 2, "b": 5, "c": 9}

In [5]:
# run.py
ampl.solve(solver="highs")
print("x =", ampl.var["x"].value(), " worst =", ampl.var["worst"].value())

HiGHS 1.14.0: optimal solution; objective 3.5
0 simplex iterations
0 barrier iterations
x = 5.5  worst = 3.5


### Pure AMPL: data.dat and model.run

The answer is the midpoint of the extreme targets, 5.5, and `worst` equals the true largest
deviation, `abs(5.5 - 2) = abs(5.5 - 9) = 3.5`.

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set I := a b c;
param target := a 2  b 5  c 9;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display x, worst;

HiGHS 1.14.0: optimal solution; objective 3.5
0 simplex iterations
0 barrier iterations
x = 5.5
worst = 3.5

